# Análisis de expresión diferencial en Leucemia Linfoide Aguda (LLA)
## Subtipos moleculares **BCR/ABL vs NEG** con limma en InMoose

**Máster en Bioinformática — Análisis de datos de microarrays**

Este notebook ejecuta en Google Colab el pipeline completo de análisis de expresión diferencial
con **limma** (implementación Python en el paquete `inmoose`):

1. Instalación del entorno
2. Carga de datos de expresión (`exprs.csv`) y metadatos (`pData.csv`)
3. Filtrado de muestras BCR/ABL vs NEG
4. Matriz de diseño y modelado con limma (`lmFit` → `contrasts.fit` → `eBayes` → `topTable`)
5. Visualización: MA plot, Volcano plot y Heatmaps
6. Exportación y descarga de resultados

> **Nota:** la celda de instalación puede tardar varios minutos (inmoose se compila desde
> el código fuente). Ejecuta las celdas en orden con **Entorno de ejecución → Ejecutar todas**.

## 0. Instalación de `inmoose`

`inmoose` no distribuye binarios precompilados: `pip` lo compila desde el código fuente,
por lo que **esta celda puede tardar entre 5 y 10 minutos**. Solo hay que ejecutarla una vez por sesión.
Si la sesión se reinicia, vuelve a ejecutarla.

In [ ]:
# ============================================================================
# CELDA 0: Instalación de inmoose (solo una vez por sesión)
# ============================================================================
!pip install -q inmoose==0.8.1

import inmoose
print(f"✓ inmoose {inmoose.__version__} instalado correctamente")

## 1. Importación de librerías

In [ ]:
# ============================================================================
# CELDA 1: Importación de librerías
# ============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Suprimir advertencias para output más limpio
warnings.filterwarnings('ignore')

# Importar limma desde inmoose (implementación Python de limma de R)
from inmoose import limma
import patsy  # Para crear matrices de diseño con fórmulas R-style

print("✓ Librerías importadas correctamente")

## 2. Carga de los datos de la práctica

Necesitas los dos archivos descargados de la plataforma:

| Archivo | Contenido |
|---|---|
| `exprs.csv` | Matriz de expresión (genes × muestras) |
| `pData.csv` | Metadatos de las muestras (subtipo molecular, datos clínicos) |

**Elige una de las tres opciones** cambiando el valor de `MODO` en la celda siguiente:

- `"SUBIR"`: sube los dos CSV desde tu ordenador (los archivos se pierden al cerrar la sesión).
- `"DRIVE"`: los archivos están en una carpeta de tu Google Drive (ajusta la ruta).
- `"URL"`: descarga directa desde una URL, p. ej. el repositorio GitHub del curso (ajusta `URL_BASE`).

In [ ]:
# ============================================================================
# CELDA 2: Obtención de los datos — por defecto se descargan del repo del curso
# (opciones alternativas: "SUBIR" para subida manual, "DRIVE" para Google Drive)
# ============================================================================

MODO = "URL"   # "URL" | "SUBIR" | "DRIVE"

if MODO == "SUBIR":
    # Opción A: subir exprs.csv y pData.csv desde el ordenador
    from google.colab import files
    print("Sube exprs.csv y pData.csv (puedes seleccionar los dos a la vez):")
    files.upload()
    DATA_DIR = Path("/content")

elif MODO == "DRIVE":
    # Opción B: archivos en una carpeta de Google Drive (ajusta la ruta)
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/LLA_microarray")  # <-- AJUSTA ESTA RUTA

elif MODO == "URL":
    # Opción C: descarga desde URL (p. ej. GitHub raw del repositorio del curso)
    import urllib.request
    DATA_DIR = Path("/content/datos_lla")
    DATA_DIR.mkdir(exist_ok=True)
    URL_BASE = "https://raw.githubusercontent.com/TU_USUARIO/transcriptomica-master/main/practicas/01-microarrays-lla/datos/"  # <-- Sustituye TU_USUARIO por tu usuario de GitHub
    urllib.request.urlretrieve(URL_BASE + "exprs.csv", DATA_DIR / "exprs.csv")
    urllib.request.urlretrieve(URL_BASE + "pData.csv", DATA_DIR / "pData.csv")

else:
    raise ValueError("MODO debe ser 'SUBIR', 'DRIVE' o 'URL'")

expr_path = DATA_DIR / "exprs.csv"
pdata_path = DATA_DIR / "pData.csv"
assert expr_path.exists(), f"No se encuentra {expr_path}"
assert pdata_path.exists(), f"No se encuentra {pdata_path}"
print(f"✓ Datos localizados en: {DATA_DIR}")

## 3. Configuración del análisis

Parámetros estadísticos y directorio de salida. Los resultados se guardan en
`/content/limma_results_LLA` y se descargan como ZIP al final del notebook.

In [ ]:
# ============================================================================
# CELDA 3: Configuración de parámetros y rutas
# ============================================================================

# --- Directorio de salida para resultados ---
OUT_DIR = Path("/content/limma_results_LLA")
OUT_DIR.mkdir(exist_ok=True)

# --- Umbrales estadísticos para significancia ---
P_VALUE_THRESHOLD = 0.05    # FDR < 5% (corrección múltiple de Benjamini-Hochberg)
LOG_FC_THRESHOLD = 0.5      # |log2 Fold Change| > 0.5 (≈ 1.4x cambio en expresión)

print(f"✓ Configuración: FDR < {P_VALUE_THRESHOLD}, log2FC > |{LOG_FC_THRESHOLD}|")

## 4. Carga y alineación de datos

Leemos la matriz de expresión y los metadatos, y nos aseguramos de que **las mismas muestras**
están presentes en ambos archivos (intersección de identificadores).

In [ ]:
# ============================================================================
# CELDA 4: Carga y preparación de datos
# ============================================================================

print("\n📊 CARGANDO Y ALINEANDO DATOS LLA...")

# Cargar matriz de expresión (genes × muestras)
datos_expresion_full = pd.read_csv(expr_path, index_col=0)

# Cargar metadatos (información de muestras: grupos, características)
design_matrix_full = pd.read_csv(pdata_path, index_col=0)

# --- ALINEAMIENTO CRÍTICO: Asegurar que muestras coincidan en ambos archivos ---
common_samples = datos_expresion_full.columns.intersection(design_matrix_full.index)
datos_expresion_full = datos_expresion_full[common_samples]
design_matrix_full = design_matrix_full.loc[common_samples]

print(f"✓ {len(common_samples)} muestras alineadas correctamente")
print(f"  Matriz de expresión: {datos_expresion_full.shape[0]} genes × {datos_expresion_full.shape[1]} muestras")

## 5. Filtrado de muestras BCR/ABL vs NEG

Seleccionamos únicamente los dos subtipos moleculares de la comparación.
**NEG actúa como grupo de referencia** del modelo.

In [ ]:
# ============================================================================
# CELDA 5: Filtrado de muestras BCR/ABL vs NEG
# ============================================================================

print("\n🔬 FILTRANDO MUESTRAS PARA BCR/ABL vs NEG...")

# Definir grupos de interés para la comparación
grupos_interes = ['BCR/ABL', 'NEG']

# Filtrar metadatos para mantener solo muestras de estos grupos
design_lla = design_matrix_full[design_matrix_full['mol.biol'].isin(grupos_interes)].copy()

# Renombrar columna para evitar problemas con puntos en nombres
design_lla = design_lla.rename(columns={'mol.biol': 'mol_biol'})

# Obtener IDs de muestras filtradas
muestras_lla = design_lla.index

# Filtrar matriz de expresión para mantener solo muestras de interés
datos_expresion = datos_expresion_full[muestras_lla]

# Convertir a matriz numpy para limma (requerimiento de inmoose)
expr_matrix = datos_expresion.values.astype(float)

# --- PREPARAR VARIABLE CATEGÓRICA ---
design_lla['mol_biol'] = design_lla['mol_biol'].astype('category').cat.reorder_categories(
    ['NEG', 'BCR/ABL'], ordered=True  # NEG será el grupo de referencia
)

print(f"✓ {len(muestras_lla)} muestras filtradas: {design_lla['mol_biol'].value_counts().to_dict()}")

## 6. Matriz de diseño y análisis con limma

El flujo de limma tiene cuatro pasos:

1. **`lmFit`** — ajusta un modelo lineal por gen: `Expresión = β₀ + β₁·grupo + ε`
2. **`contrasts.fit`** — define el contraste de interés (BCR/ABL vs NEG)
3. **`eBayes`** — moderación bayesiana empírica: "pide prestada" información entre genes
   para estabilizar las varianzas (clave con pocas réplicas)
4. **`topTable`** — tabla de resultados ordenada por significancia, con corrección
   de Benjamini-Hochberg (`fdr_bh`) para controlar los falsos positivos

In [ ]:
# ============================================================================
# CELDA 6: Matriz de diseño y análisis diferencial con limma
# ============================================================================

print("\n⚙️ PREPARANDO MATRIZ DE DISEÑO CON PATSY...")

# Fórmula estilo R: Treatment coding con NEG como referencia
formula = "~ C(mol_biol, Treatment(reference='NEG'))"

# Crear matriz de diseño usando patsy (similar a model.matrix en R)
design_df = patsy.dmatrix(formula, design_lla, return_type="dataframe")
design = patsy.DesignMatrix(design_df.values, design_df.design_info)
coef_names = design.design_info.column_names

print(f"✓ Matriz de diseño creada: {coef_names}")

# --- ANÁLISIS DE EXPRESIÓN DIFERENCIAL CON LIMMA ---
print("\n🧬 INICIANDO ANÁLISIS LIMMA...")

# 1. AJUSTAR MODELO LINEAL PARA CADA GEN
fit = limma.lmFit(expr_matrix, design)

# 2. DEFINIR CONTRASTE DE INTERÉS (BCR/ABL vs NEG)
bcr_name = [c for c in coef_names if 'BCR' in c or 'BCR/ABL' in c or 'BCR_ABL' in c][0]
contrast_df = pd.DataFrame(0, index=coef_names, columns=["BCRABL_vs_NEG"])
contrast_df.loc[bcr_name, "BCRABL_vs_NEG"] = 1

# 3. APLICAR CONTRASTES AL MODELO
fit2 = limma.contrasts_fit(fit, contrast_df)

# 4. MODERACIÓN BAYESIANA (eBayes) - PASO CLAVE
fit2 = limma.eBayes(fit2)

# 5. EXTRAER RESULTADOS
results_df = pd.DataFrame(limma.topTable(fit2, coef="BCRABL_vs_NEG", number=np.inf, adjust_method="fdr_bh"))
results_df.index = datos_expresion.index  # Restaurar nombres de genes

print("✓ Análisis limma completado")
print(f"  Columnas de la tabla de resultados: {results_df.columns.tolist()}")

## 7. Clasificación de genes por significancia

Detectamos automáticamente las columnas de p-valor ajustado y log2FC (los nombres varían
entre versiones de inmoose), y clasificamos los genes en **sobre-expresados**,
**infra-expresados** y **no significativos** según los umbrales FDR < 0.05 y |log2FC| > 0.5.

In [ ]:
# ============================================================================
# CELDA 7: Detección de columnas y clasificación de genes
# ============================================================================

print("\n📈 ANALIZANDO RESULTADOS...")

def detect_cols_robust(res_df):
    """
    Función robusta para detectar automáticamente nombres de columnas
    en diferentes versiones de limma/inmoose
    """
    # Buscar columna de p-valores ajustados
    p_candidates = ['adj.P.Val', 'FDR', 'padj']
    p_col = next((col for col in p_candidates if col in res_df.columns), None)

    if p_col is None:
        p_candidates_generic = [col for col in res_df.columns if any(x in col.lower() for x in ['adj', 'fdr', 'padj', 'p.adj'])]
        p_col = p_candidates_generic[0] if p_candidates_generic else None

    if p_col is None:
        raise KeyError("No se pudo identificar la columna de p-valores ajustados.")

    # Buscar columna de log Fold Change
    log_candidates = ['logFC', 'log2FC']
    log_col = next((col for col in log_candidates if col in res_df.columns), None)

    if log_col is None:
        log_candidates_generic = [col for col in res_df.columns if 'FC' in col or 'log' in col.lower()]
        log_col = log_candidates_generic[0] if log_candidates_generic else None

    if log_col is None:
        raise KeyError(f"No se pudo identificar la columna de Log Fold Change. Columnas disponibles: {res_df.columns.tolist()}")

    return p_col, log_col

# Detectar nombres de columnas automáticamente
p_col, log_col = detect_cols_robust(results_df)
print(f"✓ Columnas detectadas: p-valor='{p_col}', logFC='{log_col}'")

# --- CLASIFICAR GENES POR SIGNIFICANCIA ---
genes_significativos = results_df[results_df[p_col] < P_VALUE_THRESHOLD]
genes_sobre = genes_significativos[genes_significativos[log_col] > LOG_FC_THRESHOLD]
genes_infra = genes_significativos[genes_significativos[log_col] < -LOG_FC_THRESHOLD]

# Añadir columna de categoría para visualizaciones
results_df['Categoria'] = 'No significativo'
results_df.loc[(results_df[p_col] < P_VALUE_THRESHOLD) & (results_df[log_col] > LOG_FC_THRESHOLD), 'Categoria'] = 'Sobre-expresado'
results_df.loc[(results_df[p_col] < P_VALUE_THRESHOLD) & (results_df[log_col] < -LOG_FC_THRESHOLD), 'Categoria'] = 'Infra-expresado'

# Paleta de colores para visualizaciones
paleta = {'No significativo': 'gray', 'Sobre-expresado': 'red', 'Infra-expresado': 'blue'}

print(f"✓ Genes clasificados: {len(genes_sobre)} sobre-expresados, {len(genes_infra)} infra-expresados")

## 8. MA plot y Volcano plot

- **MA plot:** log2FC frente a la expresión media — detecta sesgos dependientes de la intensidad.
- **Volcano plot:** log2FC frente a −log₁₀(FDR) — resume magnitud y significancia a la vez.

In [ ]:
# ============================================================================
# CELDA 8: MA Plot y Volcano Plot
# ============================================================================

print("\n📉 CREANDO MA PLOT...")
sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))

for categoria, color in paleta.items():
    subset = results_df[results_df['Categoria'] == categoria]
    ax.scatter(subset['AveExpr'], subset[log_col], c=color, alpha=0.6, s=15,
               label=f'{categoria} (n={len(subset)})')

ax.axhline(0, color='black', linestyle='-', alpha=0.5)
ax.axhline(LOG_FC_THRESHOLD, color='black', linestyle='--', alpha=0.5)
ax.axhline(-LOG_FC_THRESHOLD, color='black', linestyle='--', alpha=0.5)

ax.set_xlabel('Expresión Media (AveExpr)', fontsize=12)
ax.set_ylabel('log2 Fold Change', fontsize=12)
ax.set_title('MA Plot - Expresión Diferencial (BCR/ABL vs NEG)', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
fig.savefig(OUT_DIR / "MA_Plot_BCRABL_vs_NEG.png", dpi=150)
plt.show()

print("\n🎨 CREANDO VOLCANO PLOT...")
fig, ax = plt.subplots(figsize=(10, 8))

for categoria, color in paleta.items():
    subset = results_df[results_df['Categoria'] == categoria]
    ax.scatter(subset[log_col], -np.log10(subset[p_col]), c=color, alpha=0.6, s=15,
               label=f'{categoria} (n={len(subset)})')

ax.axhline(-np.log10(P_VALUE_THRESHOLD), color='black', linestyle='--', alpha=0.7)
ax.axvline(LOG_FC_THRESHOLD, color='black', linestyle='--', alpha=0.7)
ax.axvline(-LOG_FC_THRESHOLD, color='black', linestyle='--', alpha=0.7)

ax.set_xlabel('log2 Fold Change (BCR/ABL vs NEG)', fontsize=12)
ax.set_ylabel(f'-log10({p_col})', fontsize=12)
ax.set_title('Volcano Plot - Expresión Diferencial (BCR/ABL vs NEG)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_DIR / "volcano_BCRABL_vs_NEG.png", dpi=150)
plt.show()

## 9. Heatmaps de los genes más significativos

Dos representaciones con Z-score por gen:

1. **Clustermap principal:** top 40 genes más significativos, con agrupamiento (clustering) de genes.
2. **Heatmap UP/DOWN:** top genes sobre-expresados e infra-expresados por separado,
   manteniendo el orden de regulación.

In [ ]:
# ============================================================================
# CELDA 9: Heatmaps
# ============================================================================

print("\n🔥 CREANDO HEATMAPS ESENCIALES...")

if len(genes_significativos) > 0:

    grupo_colores = {'BCR/ABL': '#FF8C00', 'NEG': '#008000'}  # Naranja y Verde

    # ========================================================================
    # PREPARACIÓN DE DATOS PARA HEATMAPS
    # ========================================================================

    N_TOP_GENES = min(40, len(genes_significativos))
    top_genes_indices = genes_significativos.nsmallest(N_TOP_GENES, p_col).index
    heatmap_data = datos_expresion.loc[top_genes_indices]

    # Ordenar muestras por grupo para mejor visualización
    design_heatmap = design_lla.copy()
    design_heatmap['mol_biol'] = pd.Categorical(design_heatmap['mol_biol'],
                                               categories=['NEG', 'BCR/ABL'], ordered=True)
    design_heatmap = design_heatmap.sort_values('mol_biol')
    muestras_ordenadas = design_heatmap.index.tolist()
    heatmap_data = heatmap_data[muestras_ordenadas]

    # Normalización Z-score por filas (genes)
    heatmap_zscore = heatmap_data.apply(lambda x: (x - x.mean()) / x.std(), axis=1)
    heatmap_zscore = heatmap_zscore.dropna()

    print(f"✓ Datos preparados: {len(heatmap_zscore)} genes × {len(muestras_ordenadas)} muestras")

    # ========================================================================
    # HEATMAP 1: EXPRESIÓN GENERAL (Clustermap principal)
    # ========================================================================

    print("\n📊 Creando Heatmap Principal...")

    col_colors = design_heatmap['mol_biol'].map(grupo_colores)

    g = sns.clustermap(
        heatmap_zscore,
        cmap='RdBu_r',
        center=0,
        vmin=-2, vmax=2,
        row_cluster=True,
        col_cluster=False,
        col_colors=col_colors,
        yticklabels=True,
        xticklabels=False,
        figsize=(12, 10),
        dendrogram_ratio=0.12,
        cbar_pos=(0.02, 0.87, 0.03, 0.1),
        linewidths=0,
        cbar_kws={'label': 'Z-score (σ)'}
    )

    g.ax_heatmap.set_ylabel('Genes', fontsize=11, fontweight='bold')
    g.ax_heatmap.set_xlabel('Muestras', fontsize=11, fontweight='bold')
    g.ax_heatmap.tick_params(axis='y', labelsize=7)

    legend_grupos = [
        Patch(facecolor=grupo_colores['NEG'], edgecolor='black',
              label=f"NEG (n={(design_heatmap['mol_biol'] == 'NEG').sum()})"),
        Patch(facecolor=grupo_colores['BCR/ABL'], edgecolor='black',
              label=f"BCR/ABL (n={(design_heatmap['mol_biol'] == 'BCR/ABL').sum()})")
    ]

    g.ax_col_dendrogram.legend(handles=legend_grupos, title='Subtipo Molecular',
                              loc='center left', bbox_to_anchor=(0.03, 0.5),
                              frameon=True, fontsize=10, title_fontsize=11)

    g.fig.suptitle(f'Top {len(heatmap_zscore)} Genes Diferencialmente Expresados\nBCR/ABL vs NEG',
                   fontsize=14, fontweight='bold', y=0.97)

    plt.savefig(OUT_DIR / "heatmap_principal_BCRABL_vs_NEG.png", dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Heatmap principal guardado")

    # ========================================================================
    # HEATMAP 2: GENES UP vs DOWN SEPARADOS
    # ========================================================================

    print("\n↕️ Creando Heatmap de Genes UP vs DOWN...")

    N_MAX_UP_DOWN = 25
    genes_up_top = genes_sobre.nsmallest(min(N_MAX_UP_DOWN, len(genes_sobre)), p_col).index
    genes_down_top = genes_infra.nsmallest(min(N_MAX_UP_DOWN, len(genes_infra)), p_col).index

    if len(genes_up_top) > 0 and len(genes_down_top) > 0:
        genes_combined = list(genes_up_top) + list(genes_down_top)
        heatmap_combined = datos_expresion.loc[genes_combined, muestras_ordenadas]
        heatmap_combined_z = heatmap_combined.apply(lambda x: (x - x.mean()) / x.std(), axis=1)
        heatmap_combined_z = heatmap_combined_z.dropna()

        num_genes_up = len(genes_up_top.intersection(heatmap_combined_z.index))
        num_genes_down = len(genes_down_top.intersection(heatmap_combined_z.index))

        col_colors = design_heatmap['mol_biol'].map(grupo_colores)

        g = sns.clustermap(
            heatmap_combined_z,
            cmap='RdBu_r',
            center=0,
            vmin=-2.5, vmax=2.5,
            row_cluster=False,
            col_cluster=False,
            col_colors=col_colors,
            yticklabels=True,
            xticklabels=False,
            figsize=(14, 10),
            dendrogram_ratio=0.1,
            cbar_pos=(0.02, 0.83, 0.03, 0.12),
            linewidths=0,
            cbar_kws={'label': 'Z-score (σ)'}
        )

        g.ax_heatmap.set_ylabel('Genes', fontsize=11, fontweight='bold')
        g.ax_heatmap.set_xlabel('Muestras', fontsize=11, fontweight='bold')
        g.ax_heatmap.tick_params(axis='y', labelsize=7)

        legend_grupos = [
            Patch(facecolor=grupo_colores['NEG'], edgecolor='black',
                  label=f"NEG (n={(design_heatmap['mol_biol'] == 'NEG').sum()})"),
            Patch(facecolor=grupo_colores['BCR/ABL'], edgecolor='black',
                  label=f"BCR/ABL (n={(design_heatmap['mol_biol'] == 'BCR/ABL').sum()})")
        ]

        legend_reg = [
            Patch(facecolor='red', edgecolor='black', label=f'↑ UP (n={num_genes_up})'),
            Patch(facecolor='blue', edgecolor='black', label=f'↓ DOWN (n={num_genes_down})')
        ]

        g.ax_col_dendrogram.legend(
            handles=legend_grupos + legend_reg,
            title='Leyenda',
            loc='center left',
            bbox_to_anchor=(0.03, 0.8),
            frameon=True,
            fontsize=10,
            title_fontsize=11
        )

        g.fig.suptitle('Top Genes Sobre e Infra-expresados\n' +
                       f'BCR/ABL vs NEG | ↑ UP: {num_genes_up} genes | ↓ DOWN: {num_genes_down} genes',
                       fontsize=14, fontweight='bold', y=0.97)

        plt.savefig(OUT_DIR / "heatmap_up_down_BCRABL_vs_NEG.png", dpi=300, bbox_inches='tight')
        plt.show()
        print("✓ Heatmap UP vs DOWN guardado (formato consistente)")

    print("\n📊 RESUMEN DE HEATMAPS GENERADOS:")
    print(f"  1. Clustermap Principal: {len(heatmap_zscore)} genes más significativos")
    if len(genes_up_top) > 0 and len(genes_down_top) > 0:
        print(f"  2. Heatmap UP/DOWN: {num_genes_up} genes UP + {num_genes_down} genes DOWN")

else:
    print("⚠️ No hay genes significativos para crear heatmaps")

print("\n--- HEATMAPS COMPLETADOS ---")

## 10. Exportación y descarga de resultados

Se generan cuatro archivos CSV (resultados completos, significativos, sobre-expresados e
infra-expresados) y todas las figuras en PNG. La última celda comprime todo en un ZIP
y lo descarga a tu ordenador.

In [ ]:
# ============================================================================
# CELDA 10: Guardar resultados
# ============================================================================

print("\n💾 GUARDANDO RESULTADOS...")

print("\n🏆 TOP 10 GENES MÁS SIGNIFICATIVOS:")
top_genes = results_df.nsmallest(10, p_col)
for i, (gen, row) in enumerate(top_genes.iterrows(), 1):
    direccion = "↑ SOBRE" if row[log_col] > 0 else "↓ INFRA"
    print(f"  {i:2d}. {gen}: logFC={row[log_col]:6.2f} {direccion} {p_col}={row[p_col]:.2e}")

# --- GUARDAR ARCHIVOS DE RESULTADOS ---
results_df.to_csv(OUT_DIR / "results_BCRABL_vs_NEG.csv")
genes_significativos.to_csv(OUT_DIR / "sig_BCRABL_vs_NEG.csv")
genes_sobre.to_csv(OUT_DIR / "up_BCRABL_vs_NEG.csv")
genes_infra.to_csv(OUT_DIR / "down_BCRABL_vs_NEG.csv")

print(f"✓ Archivos guardados en: {OUT_DIR}")
print("\n--- ANÁLISIS COMPLETADO EXITOSAMENTE ---")

In [ ]:
# ============================================================================
# CELDA 11: Comprimir y descargar todos los resultados (ZIP)
# ============================================================================

import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/limma_results_LLA", "zip", OUT_DIR)
print(f"✓ ZIP creado: {zip_path}")
files.download(zip_path)

---
## Ejercicios propuestos

1. **Umbrales estadísticos:** cambia `LOG_FC_THRESHOLD` a 1.0 y `P_VALUE_THRESHOLD` a 0.01.
   ¿Cuántos genes significativos quedan? ¿Qué compromiso hay entre sensibilidad y falsos positivos?
2. **Otro contraste:** repite el análisis comparando otro subtipo molecular frente a NEG
   (consulta primero `design_matrix_full['mol.biol'].value_counts()`).
3. **El papel de eBayes:** comenta la línea `fit2 = limma.eBayes(fit2)` y usa directamente
   `fit` en `contrasts_fit`. ¿Cómo cambian los p-valores? ¿Por qué?
4. **Anotación:** los identificadores (`1000_at`, `1007_s_at`…) son sondas Affymetrix de la
   plataforma HG-U95Av2. Busca en [NetAffx](https://www.affymetrix.com/) o BioMart a qué genes
   corresponden las 5 sondas más significativas.
5. **Interpretación biológica:** elige un gen del top 10 sobre-expresado y otro infra-expresado
   y discute su posible relación con la biología de la LLA BCR/ABL+ (fusión BCR::ABL1).